# 04 — Ingeniería de variables y construcción individual del EPBI

> **Pipeline:** 01 Auditoría → 02 Limpieza ESS → 03 Eurostat → **04 Construcción EPBI** → 05 Integración micro–macro → 06 Econometría → 07 Datos para dashboard

Con la muestra ESS ya limpia en el notebook 02 y el panel Eurostat preparado en paralelo por el notebook 03, esta etapa vuelve a la **rama individual** del proyecto. Su objetivo es construir el **Economic Perception Bias Index (EPBI)** y las variables derivadas necesarias, manteniendo al entrevistado como unidad de análisis.

No se vuelve a limpiar la ESS ni se incorpora todavía información macroeconómica. La finalidad es producir un fichero individual autocontenido que pueda integrarse con Eurostat en el notebook 05.

## Criterios de esta etapa

1. La unidad de análisis sigue siendo el **individuo ESS**.
2. La selección temporal y la limpieza de la muestra quedan heredadas del notebook 02.
3. `analysis_weight`, `psu` y `stratum` se conservan sin reconstruirlos.
4. El EPBI solo se calcula cuando `hinctnta` y `hincfel` son sustantivamente válidos.
5. Los individuos de contextos elegibles que no dispongan de ambos inputs se mantienen por trazabilidad con `epbi = NA`.
6. No se realiza ningún cruce con Eurostat.
7. No se genera ningún agregado país-ronda.
8. La única salida es `ess_epbi_micro.parquet`.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

PROCESADOS = PROJECT_ROOT / "DATOS" / "PROCESADOS"
PROCESADOS.mkdir(parents=True, exist_ok=True)

INPUT_PARQUET = PROCESADOS / "ess_clean_epbi_sample.parquet"
OUTPUT_PARQUET = PROCESADOS / "ess_epbi_micro.parquet"

print("Entrada:", INPUT_PARQUET)
print("Salida:", OUTPUT_PARQUET)

## 1. Carga y validación de la entrada

Se carga exclusivamente `ess_clean_epbi_sample.parquet`, generado en el notebook 02. Antes de construir nuevas variables se comprueba que contiene los identificadores, los inputs del EPBI, los pesos y las variables de diseño esperadas, y que la muestra sigue restringida al periodo analítico.


In [ ]:
if not INPUT_PARQUET.exists():
    raise FileNotFoundError(
        "No se encuentra ess_clean_epbi_sample.parquet. "
        "Ejecuta primero el notebook 02."
    )

ess = pd.read_parquet(INPUT_PARQUET)

required = [
    "respondent_id",
    "cntry",
    "essround",
    "survey_year",
    "hinctnta",
    "hincfel",
    "analysis_weight",
    "psu",
    "stratum",
]

missing = [var for var in required if var not in ess.columns]
if missing:
    raise KeyError(
        "Faltan variables necesarias procedentes del notebook 02: "
        + ", ".join(missing)
    )

if ess["respondent_id"].duplicated().any():
    raise ValueError("respondent_id no es único. Revisar el notebook 02.")

if (pd.to_numeric(ess["essround"], errors="coerce") < 4).any():
    raise ValueError("Se han detectado rondas anteriores a la ronda 4.")

print("Dimensiones de entrada:", ess.shape)
print("Países:", ess["cntry"].nunique())
print("Rondas:", sorted(ess["essround"].dropna().unique().tolist()))
print("Periodo:", ess["survey_year"].min(), "-", ess["survey_year"].max())

## 2. Comprobación de los inputs del EPBI

Con la entrada validada, se revisa de nuevo la disponibilidad sustantiva de `hinctnta` y `hincfel`. Este control no repite la limpieza del notebook 02: confirma que las reglas heredadas siguen siendo coherentes justo antes de calcular el índice.


In [ ]:
hinctnta = pd.to_numeric(ess["hinctnta"], errors="coerce")
hincfel = pd.to_numeric(ess["hincfel"], errors="coerce")

valid_hinctnta = hinctnta.between(1, 10)
valid_hincfel = hincfel.between(1, 4)
valid_epbi_input = valid_hinctnta & valid_hincfel

# El notebook 02 ya creó esta bandera. Si existe, comprobamos consistencia.
if "epbi_input_valid" in ess.columns:
    previous_flag = ess["epbi_input_valid"].fillna(False).astype(bool)
    n_discrepancies = int((previous_flag != valid_epbi_input).sum())

    if n_discrepancies:
        raise ValueError(
            f"Hay {n_discrepancies} discrepancias entre epbi_input_valid "
            "y los valores actuales de hinctnta/hincfel."
        )

coverage = (
    ess.groupby(["essround", "survey_year"], dropna=False)
    .agg(
        n_total=("respondent_id", "size"),
        n_hinctnta_valid=("hinctnta", lambda s: pd.to_numeric(s, errors="coerce").between(1, 10).sum()),
        n_hincfel_valid=("hincfel", lambda s: pd.to_numeric(s, errors="coerce").between(1, 4).sum()),
    )
    .reset_index()
)

coverage["n_epbi_valid"] = (
    ess.assign(_valid=valid_epbi_input)
    .groupby(["essround", "survey_year"], dropna=False)["_valid"]
    .sum()
    .to_numpy()
)

coverage["pct_epbi_valid"] = (
    100 * coverage["n_epbi_valid"] / coverage["n_total"]
).round(2)

display(coverage)

## 3. Construcción del EPBI individual

El índice compara una aproximación normalizada a la posición económica objetiva del hogar con la percepción subjetiva de su situación económica.

La posición objetiva se obtiene a partir del decil de renta del hogar:

\[
\text{Posición objetiva}_i = \frac{hinctnta_i - 1}{9}
\]

La percepción de ingresos se invierte para que los valores altos representen una situación más favorable:

\[
hincfel^{+}_i = 5 - hincfel_i
\]

y se normaliza entre 0 y 1:

\[
\text{Posición subjetiva}_i = \frac{hincfel^{+}_i - 1}{3}
\]

Finalmente:

\[
EPBI_i = \text{Posición subjetiva}_i - \text{Posición objetiva}_i
\]

La interpretación es:

- `EPBI > 0`: la percepción es relativamente más favorable que la posición objetiva aproximada;
- `EPBI < 0`: la percepción es relativamente menos favorable;
- `EPBI ≈ 0`: existe mayor alineación entre ambas dimensiones.

El índice se calcula a nivel individual y **no se redefine mediante el peso de análisis**.


In [ ]:
epbi = ess.copy()

# Validación sustantiva explícita de los dos inputs.
hinctnta_num = pd.to_numeric(epbi["hinctnta"], errors="coerce")
hinctnta_num = hinctnta_num.where(hinctnta_num.between(1, 10))

hincfel_num = pd.to_numeric(epbi["hincfel"], errors="coerce")
hincfel_num = hincfel_num.where(hincfel_num.between(1, 4))

# Posición objetiva: decil ESS armonizado a [0, 1].
epbi["objective_position"] = (hinctnta_num - 1) / 9

# Posición subjetiva: percepción invertida y armonizada a [0, 1].
epbi["hincfel_positive"] = 5 - hincfel_num
epbi["subjective_position"] = (epbi["hincfel_positive"] - 1) / 3

# EPBI individual.
epbi["epbi"] = (
    epbi["subjective_position"]
    - epbi["objective_position"]
)

epbi["epbi_abs"] = epbi["epbi"].abs()
epbi["valid_epbi"] = epbi["epbi"].notna()

# Categorías acordadas:
#   EPBI < -0.15          → subestimación
#   -0.15 <= EPBI <= .15 → ajuste
#   EPBI > 0.15           → sobreestimación
epbi["epbi_category"] = pd.Series(pd.NA, index=epbi.index, dtype="string")

epbi.loc[epbi["epbi"] < -0.15, "epbi_category"] = "subestimacion"
epbi.loc[epbi["epbi"].between(-0.15, 0.15, inclusive="both"), "epbi_category"] = "ajuste"
epbi.loc[epbi["epbi"] > 0.15, "epbi_category"] = "sobreestimacion"

display(
    epbi[
        [
            "objective_position",
            "subjective_position",
            "epbi",
            "epbi_abs",
        ]
    ].describe()
)

print("EPBI válido:", int(epbi["valid_epbi"].sum()))
print(
    "EPBI no calculable por falta de uno o ambos inputs:",
    int((~epbi["valid_epbi"]).sum()),
)

## 4. Variables auxiliares individuales

Una vez construido el índice, se generan variables derivadas que facilitan tanto el análisis como la visualización posterior, por ejemplo el valor absoluto del EPBI, su clasificación por categorías y agrupaciones descriptivas de edad.


In [ ]:
# Grupo de edad: variable de presentación/análisis.
if "agea" in epbi.columns:
    age = pd.to_numeric(epbi["agea"], errors="coerce")

    epbi["age_group"] = pd.cut(
        age,
        bins=[14, 24, 34, 44, 54, 64, 74, np.inf],
        labels=[
            "15_24",
            "25_34",
            "35_44",
            "45_54",
            "55_64",
            "65_74",
            "75_plus",
        ],
    )

# Índice descriptivo de confianza institucional.
# Se calcula únicamente a partir de las variables efectivamente disponibles.
trust_vars = [
    var
    for var in ["ppltrst", "trstprl", "trstplt", "stfdem"]
    if var in epbi.columns
]

if trust_vars:
    trust_numeric = epbi[trust_vars].apply(pd.to_numeric, errors="coerce")
    epbi["institutional_trust_index"] = trust_numeric.mean(axis=1, skipna=True)
    epbi.loc[trust_numeric.notna().sum(axis=1).eq(0), "institutional_trust_index"] = np.nan

print("Variables auxiliares creadas:")
print(" - age_group:", "age_group" in epbi.columns)
print(" - institutional_trust_index:", "institutional_trust_index" in epbi.columns)

## 5. Pesos y diseño muestral

`analysis_weight`, `psu` y `stratum` proceden del notebook 02 y se mantienen sin modificaciones. En esta sección únicamente se comprueba su disponibilidad y coherencia.

El peso se utilizará posteriormente en descriptivos ponderados y WLS. La decisión sobre la inferencia y el no uso de PSU en la especificación final se documentará en el notebook 06.


In [ ]:
weight = pd.to_numeric(epbi["analysis_weight"], errors="coerce")

invalid_weight = weight.notna() & (~np.isfinite(weight) | (weight <= 0))

if invalid_weight.any():
    raise ValueError(
        f"Se han encontrado {int(invalid_weight.sum())} pesos no positivos o no finitos."
    )

weight_check = pd.DataFrame({
    "indicador": [
        "casos totales",
        "analysis_weight válido",
        "analysis_weight ausente",
        "psu disponible",
        "stratum disponible",
    ],
    "n": [
        len(epbi),
        int(weight.notna().sum()),
        int(weight.isna().sum()),
        int(epbi["psu"].notna().sum()),
        int(epbi["stratum"].notna().sum()),
    ],
})

display(weight_check)

if "analysis_weight_source" in epbi.columns:
    display(
        epbi["analysis_weight_source"]
        .fillna("sin_peso_valido")
        .value_counts(dropna=False)
        .rename_axis("fuente_peso")
        .reset_index(name="n")
    )

## 6. Controles finales

Antes de exportar se verifica la unicidad individual, el periodo analítico, los rangos de las posiciones normalizadas, el intervalo del EPBI y la coherencia entre `valid_epbi` y la disponibilidad real del índice.


In [ ]:
valid_epbi = epbi["epbi"].dropna()

checks = pd.DataFrame([
    {
        "control": "respondent_id único",
        "resultado": "OK" if not epbi["respondent_id"].duplicated().any() else "REVISAR",
    },
    {
        "control": "Rondas analíticas desde R4",
        "resultado": "OK" if (pd.to_numeric(epbi["essround"], errors="coerce") >= 4).all() else "REVISAR",
    },
    {
        "control": "objective_position dentro de [0,1]",
        "resultado": "OK" if epbi["objective_position"].dropna().between(0, 1).all() else "REVISAR",
    },
    {
        "control": "subjective_position dentro de [0,1]",
        "resultado": "OK" if epbi["subjective_position"].dropna().between(0, 1).all() else "REVISAR",
    },
    {
        "control": "EPBI dentro de [-1,1]",
        "resultado": "OK" if valid_epbi.between(-1, 1).all() else "REVISAR",
    },
    {
        "control": "valid_epbi coherente con epbi",
        "resultado": "OK" if epbi["valid_epbi"].equals(epbi["epbi"].notna()) else "REVISAR",
    },
    {
        "control": "Categoría EPBI solo cuando EPBI es válido",
        "resultado": "OK" if epbi.loc[epbi["epbi"].isna(), "epbi_category"].isna().all() else "REVISAR",
    },
    {
        "control": "analysis_weight no se reconstruye en 04",
        "resultado": "OK",
    },
])

display(checks)

if (checks["resultado"] != "OK").any():
    raise ValueError("Alguno de los controles finales del notebook 04 requiere revisión.")

category_distribution = (
    epbi.loc[epbi["valid_epbi"], "epbi_category"]
    .value_counts(dropna=False)
    .rename_axis("categoria")
    .reset_index(name="n")
)

category_distribution["pct"] = (
    100 * category_distribution["n"] / category_distribution["n"].sum()
).round(2)

display(category_distribution)

## 7. Exportación

Con el EPBI validado, se guarda un único dataset individual enriquecido. No se generan CSV, Pickle, Excel, metadatos ni agregados país-ronda, porque la siguiente etapa necesita conservar la granularidad individual.


In [ ]:
epbi.to_parquet(OUTPUT_PARQUET, index=False)

print("Dataset EPBI individual guardado:")
print(" -", OUTPUT_PARQUET)
print("Filas:", len(epbi))
print("Columnas:", epbi.shape[1])
print("EPBI válidos:", int(epbi["valid_epbi"].sum()))

## 8. Resultado de la fase 04 y continuidad

La salida es:

`DATOS/PROCESADOS/ess_epbi_micro.parquet`

Cada fila sigue representando un **individuo ESS**. Respecto a la salida del notebook 02, se añaden:

- `objective_position`
- `hincfel_positive`
- `subjective_position`
- `epbi`
- `epbi_abs`
- `valid_epbi`
- `epbi_category`
- `age_group`, cuando `agea` está disponible;
- `institutional_trust_index`, cuando están disponibles sus componentes.

`analysis_weight`, `analysis_weight_source`, `psu` y `stratum` se mantienen tal como fueron definidos durante la limpieza.

En este punto las dos ramas están listas: el notebook 03 aporta el **contexto país-año** y el notebook 04 aporta el **panel individual con EPBI**. El **notebook 05** será el punto de convergencia.
